In [1]:
library("xgboost")
library("Matrix")
library('Ckmeans.1d.dp')
library('lightgbm')



载入程辑包：‘lightgbm’


The following object is masked from ‘package:xgboost’:

    slice




## Read data and process data labels

In [2]:
time_matrix <- matrix(0,ncol = 3, nrow =4)
colnames(time_matrix) <- c("user_time", "system_time", "elapsed_time")
start_time = Sys.time()

In [3]:
data=read.csv('Weekly_features_matrix.csv')
data=data[,2:dim(data)[2]]

In [4]:
dim(data)

[1] 359  23

In [5]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9939369,6.320946e-11,36.725978,23.3448845,0.9600554,6.8563381,0.12476627,⋯,0.06449135,-0.45466261,0.2879798,2179,0.13012409,0.13012409,0.13012409,0,0,0
2,1,0,1,0.9946734,6.529056e-11,34.006351,19.8560557,0.9377499,5.8511863,0.09129043,⋯,0.16413331,-0.36322842,0.3123634,1710,0.11726213,0.11726213,0.11726213,0,0,0
3,1,0,1,0.9991252,8.840224e-13,44.955510,11.0485829,0.6469218,2.6893773,0.02284238,⋯,2.84368811,-0.07717025,2.4288344,2178,0.11671782,0.11671782,0.11671782,0,0,0
4,1,0,1,0.4159002,3.556671e-07,5.567640,-13.7061999,0.8181018,2.1703527,0.72989048,⋯,0.09272949,-0.55370413,0.3546872,2597,0.09217238,0.09217238,0.09217238,0,0,0
5,1,0,1,0.2103179,9.307235e-06,-12.975059,-1.0188984,0.6064015,0.7164726,0.87229229,⋯,0.10934385,-0.57273878,0.3857296,1603,0.07539105,0.07539105,0.07539105,0,0,0
6,1,0,1,0.4338204,1.481087e-06,6.188379,0.3520556,0.5692734,0.6017905,0.78743719,⋯,0.11337659,-0.56140364,0.3372221,1602,0.13123631,0.13123631,0.13123631,0,0,0


In [6]:
dlist= load('Weekly_FForma_datalist.RData')
datalist=eval(parse(text = dlist ))
res=datalist[[1]]
MASE=res[,,,6]
m=5

In [7]:
dim(MASE)

[1] 137   5   4

In [8]:
whichmin<-function(x){
    minx=min(x[x>0])
    loc=which(x==minx)[1]-1
    loc
}

meanunique=function(x)
    {
    mean(unique(x))
}

In [9]:
nanum=c()
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    a=apply(count,1,min)
    if(sum(is.na(a))>0)
        {
        nanum=append(nanum,i)
    }
    }

In [10]:
nanum

NULL

In [11]:
realbestmin=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    min_value=apply(count,1,min)
    if (max(min_value,na.rm = TRUE)==0){
        realbestmin[i,]=0
        }
    else
        {
        min_value[is.na(min_value)]=100 
        realbestmin[i,]= whichmin(min_value)
    }
    }

In [12]:
table(realbestmin)

realbestmin
 0  1  2  3  4 
20 12 24 29 52 

In [13]:
realbestmean=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    mean_value=apply(count,1,meanunique)
    if (max(mean_value,na.rm = TRUE)==0){
        realbestmean[i,]=0
        }
    else
        {
        mean_value[is.na(mean_value)]=100 
        realbestmean[i,]= whichmin(mean_value)
    }
    }

In [14]:
realbestmean

4
4
1
1
2
4
4
3
3
0
0


In [15]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9939369,6.320946e-11,36.725978,23.3448845,0.9600554,6.8563381,0.12476627,⋯,0.06449135,-0.45466261,0.2879798,2179,0.13012409,0.13012409,0.13012409,0,0,0
2,1,0,1,0.9946734,6.529056e-11,34.006351,19.8560557,0.9377499,5.8511863,0.09129043,⋯,0.16413331,-0.36322842,0.3123634,1710,0.11726213,0.11726213,0.11726213,0,0,0
3,1,0,1,0.9991252,8.840224e-13,44.955510,11.0485829,0.6469218,2.6893773,0.02284238,⋯,2.84368811,-0.07717025,2.4288344,2178,0.11671782,0.11671782,0.11671782,0,0,0
4,1,0,1,0.4159002,3.556671e-07,5.567640,-13.7061999,0.8181018,2.1703527,0.72989048,⋯,0.09272949,-0.55370413,0.3546872,2597,0.09217238,0.09217238,0.09217238,0,0,0
5,1,0,1,0.2103179,9.307235e-06,-12.975059,-1.0188984,0.6064015,0.7164726,0.87229229,⋯,0.10934385,-0.57273878,0.3857296,1603,0.07539105,0.07539105,0.07539105,0,0,0
6,1,0,1,0.4338204,1.481087e-06,6.188379,0.3520556,0.5692734,0.6017905,0.78743719,⋯,0.11337659,-0.56140364,0.3372221,1602,0.13123631,0.13123631,0.13123631,0,0,0


In [16]:
set.seed(100)
index = sample(3,dim(data)[1],replace = TRUE,prob=c(0.3,0.4,0.3))

In [17]:
train_data=data[index==2,]
test_data=data[index==3,]
train_label_min=realbestmin
train_label_mean=realbestmean


In [18]:
dim(data)

[1] 359  23

In [19]:
dim(train_data)

[1] 137  23

In [20]:
length(realbestmin)

[1] 137

In [21]:
realbestmean

4
4
1
1
2
4
4
3
3
0
0


In [22]:
head(train_data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.99393691,6.320946e-11,36.7259783,23.344885,0.9600554,6.856338,0.12476627,⋯,0.06449135,-0.4546626,0.2879798,2179,0.13012409,0.13012409,0.13012409,0,0,0
2,1,0,1,0.99467342,6.529056e-11,34.0063507,19.856056,0.9377499,5.851186,0.09129043,⋯,0.16413331,-0.3632284,0.3123634,1710,0.11726213,0.11726213,0.11726213,0,0,0
4,1,0,1,0.41590018,3.556671e-07,5.5676401,-13.706200,0.8181018,2.170353,0.72989048,⋯,0.09272949,-0.5537041,0.3546872,2597,0.09217238,0.09217238,0.09217238,0,0,0
8,1,0,1,0.09782747,1.600440e-06,-0.8409878,-3.329146,0.7405379,2.127181,0.82683581,⋯,0.13902275,-0.6097405,0.3962204,1623,0.10430646,0.10430646,0.10430646,0,0,0
10,1,0,1,0.95876248,1.637368e-08,23.3042393,-1.569799,0.8756885,3.238336,0.32979571,⋯,0.05755811,-0.4687462,0.3042919,934,0.03887081,0.03887081,0.03887081,0,0,0
13,1,0,1,0.97296035,1.611864e-08,25.3135326,5.425562,0.8144504,1.562147,0.14940563,⋯,0.17241185,-0.3284326,0.3229872,721,0.04600835,0.04600835,0.04600835,0,0,0


In [23]:
end_time = Sys.time()

In [24]:
time_matrix[1,]=end_time-start_time

In [25]:
end_time-start_time

Time difference of 13.45654 secs

## Target the interval where the actual error is minimum

In [26]:
start_time = Sys.time()

In [27]:
dtrain_xg_min_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_min)) 
dtrain_xg_min_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)) )

In [28]:
dtrain_lg_min_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_min))
dtrain_lg_min_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)))

In [29]:
xgb_min_reg <- xgboost(data = dtrain_xg_min_reg, nround=500)

[1]	train-rmse:1.863476 
[2]	train-rmse:1.393774 
[3]	train-rmse:1.064971 
[4]	train-rmse:0.820268 
[5]	train-rmse:0.645372 
[6]	train-rmse:0.523084 
[7]	train-rmse:0.405903 
[8]	train-rmse:0.327577 
[9]	train-rmse:0.284502 
[10]	train-rmse:0.248049 
[11]	train-rmse:0.211356 
[12]	train-rmse:0.187845 
[13]	train-rmse:0.164951 
[14]	train-rmse:0.142395 
[15]	train-rmse:0.127618 
[16]	train-rmse:0.116635 
[17]	train-rmse:0.104215 
[18]	train-rmse:0.089077 
[19]	train-rmse:0.080898 
[20]	train-rmse:0.067610 
[21]	train-rmse:0.056197 
[22]	train-rmse:0.048316 
[23]	train-rmse:0.042923 
[24]	train-rmse:0.036937 
[25]	train-rmse:0.033136 
[26]	train-rmse:0.028506 
[27]	train-rmse:0.024385 
[28]	train-rmse:0.021892 
[29]	train-rmse:0.019957 
[30]	train-rmse:0.017078 
[31]	train-rmse:0.014950 
[32]	train-rmse:0.012959 
[33]	train-rmse:0.011637 
[34]	train-rmse:0.009607 
[35]	train-rmse:0.008084 
[36]	train-rmse:0.007141 
[37]	train-rmse:0.006659 
[38]	train-rmse:0.005937 
[39]	train-rmse:0.005

Warning message in sprintf(save_name, env$iteration):
“one argument not used by format 'xgboost.model'”


In [30]:
xgb_min_cl <- xgboost(data = dtrain_xg_min_cl, nround=500, objective='multi:softmax',num_class=5)

[1]	train-merror:0.131387 
[2]	train-merror:0.051095 
[3]	train-merror:0.043796 
[4]	train-merror:0.014599 
[5]	train-merror:0.000000 
[6]	train-merror:0.000000 
[7]	train-merror:0.000000 
[8]	train-merror:0.000000 
[9]	train-merror:0.000000 
[10]	train-merror:0.000000 
[11]	train-merror:0.000000 
[12]	train-merror:0.000000 
[13]	train-merror:0.000000 
[14]	train-merror:0.000000 
[15]	train-merror:0.000000 
[16]	train-merror:0.000000 
[17]	train-merror:0.000000 
[18]	train-merror:0.000000 
[19]	train-merror:0.000000 
[20]	train-merror:0.000000 
[21]	train-merror:0.000000 
[22]	train-merror:0.000000 
[23]	train-merror:0.000000 
[24]	train-merror:0.000000 
[25]	train-merror:0.000000 
[26]	train-merror:0.000000 
[27]	train-merror:0.000000 
[28]	train-merror:0.000000 
[29]	train-merror:0.000000 
[30]	train-merror:0.000000 
[31]	train-merror:0.000000 
[32]	train-merror:0.000000 
[33]	train-merror:0.000000 
[34]	train-merror:0.000000 
[35]	train-merror:0.000000 
[36]	train-merror:0.000000 
[

Warning message in sprintf(save_name, env$iteration):
“one argument not used by format 'xgboost.model'”


In [31]:
lgb_min_reg <- lgb.train(data = dtrain_lg_min_reg, nrounds = 500)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011252 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 775
[LightGBM] [Info] Number of data points in the train set: 137, number of used features: 17
[LightGBM] [Info] Start training from score 2.591241
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

In [32]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_min_cl <- lgb.train(data = dtrain_lg_min_cl,nrounds = 500,params=params)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005863 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 775
[LightGBM] [Info] Number of data points in the train set: 137, number of used features: 17
[LightGBM] [Info] Start training from score -1.924249
[LightGBM] [Info] Start training from score -2.435074
[LightGBM] [Info] Start training from score -1.741927
[LightGBM] [Info] Start training from score -1.552685
[LightGBM] [Info] Start training from score -0.968737
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [33]:
end_time = Sys.time()
time_matrix[2,]=end_time-start_time

## Target the interval where the average error is minimum

In [34]:
start_time = Sys.time()

In [35]:
dtrain_xg_mean_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_mean)) 
dtrain_xg_mean_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)) )

In [36]:
dtrain_lg_mean_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_mean))
dtrain_lg_mean_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)))

In [37]:
xgb_mean_reg <- xgboost(data = dtrain_xg_mean_reg, nround=500)

[1]	train-rmse:1.647725 
[2]	train-rmse:1.236831 
[3]	train-rmse:0.935826 
[4]	train-rmse:0.726921 
[5]	train-rmse:0.563737 
[6]	train-rmse:0.456652 
[7]	train-rmse:0.377894 
[8]	train-rmse:0.304184 
[9]	train-rmse:0.248975 
[10]	train-rmse:0.212769 
[11]	train-rmse:0.180223 
[12]	train-rmse:0.163760 
[13]	train-rmse:0.145025 
[14]	train-rmse:0.129368 
[15]	train-rmse:0.114375 
[16]	train-rmse:0.104106 
[17]	train-rmse:0.098625 
[18]	train-rmse:0.087833 
[19]	train-rmse:0.079775 
[20]	train-rmse:0.068843 
[21]	train-rmse:0.062069 
[22]	train-rmse:0.054200 
[23]	train-rmse:0.048360 
[24]	train-rmse:0.045409 
[25]	train-rmse:0.041516 
[26]	train-rmse:0.038648 
[27]	train-rmse:0.036573 
[28]	train-rmse:0.032836 
[29]	train-rmse:0.028873 
[30]	train-rmse:0.025230 
[31]	train-rmse:0.022089 
[32]	train-rmse:0.019084 
[33]	train-rmse:0.018169 
[34]	train-rmse:0.015413 
[35]	train-rmse:0.014058 
[36]	train-rmse:0.012803 
[37]	train-rmse:0.011779 
[38]	train-rmse:0.010662 
[39]	train-rmse:0.009

Warning message in sprintf(save_name, env$iteration):
“one argument not used by format 'xgboost.model'”


In [38]:
xgb_mean_cl <- xgboost(data = dtrain_xg_mean_cl, nround=500, objective='multi:softmax',num_class=5)

[1]	train-merror:0.138686 
[2]	train-merror:0.051095 
[3]	train-merror:0.029197 
[4]	train-merror:0.014599 
[5]	train-merror:0.000000 
[6]	train-merror:0.000000 
[7]	train-merror:0.000000 
[8]	train-merror:0.000000 
[9]	train-merror:0.000000 
[10]	train-merror:0.000000 
[11]	train-merror:0.000000 
[12]	train-merror:0.000000 
[13]	train-merror:0.000000 
[14]	train-merror:0.000000 
[15]	train-merror:0.000000 
[16]	train-merror:0.000000 
[17]	train-merror:0.000000 
[18]	train-merror:0.000000 
[19]	train-merror:0.000000 
[20]	train-merror:0.000000 
[21]	train-merror:0.000000 
[22]	train-merror:0.000000 
[23]	train-merror:0.000000 
[24]	train-merror:0.000000 
[25]	train-merror:0.000000 
[26]	train-merror:0.000000 
[27]	train-merror:0.000000 
[28]	train-merror:0.000000 
[29]	train-merror:0.000000 
[30]	train-merror:0.000000 
[31]	train-merror:0.000000 
[32]	train-merror:0.000000 
[33]	train-merror:0.000000 
[34]	train-merror:0.000000 
[35]	train-merror:0.000000 
[36]	train-merror:0.000000 
[

Warning message in sprintf(save_name, env$iteration):
“one argument not used by format 'xgboost.model'”


In [39]:
lgb_mean_reg <- lgb.train(data = dtrain_lg_mean_reg,nrounds = 500)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004887 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 775
[LightGBM] [Info] Number of data points in the train set: 137, number of used features: 17
[LightGBM] [Info] Start training from score 2.094891
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

In [40]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_mean_cl <- lgb.train(data = dtrain_lg_mean_cl,params=params,nrounds = 500)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005512 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 775
[LightGBM] [Info] Number of data points in the train set: 137, number of used features: 17
[LightGBM] [Info] Start training from score -1.423473
[LightGBM] [Info] Start training from score -1.924249
[LightGBM] [Info] Start training from score -1.828938
[LightGBM] [Info] Start training from score -1.701105
[LightGBM] [Info] Start training from score -1.309063
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [41]:
end_time = Sys.time()
time_matrix[3,]=end_time-start_time

In [42]:
end_time-start_time

Time difference of 1.993198 mins

## predict

In [43]:
start_time = Sys.time()

In [44]:
alldatalgb <- lgb.Dataset(data = as.matrix(data))
alldataxgb <- xgb.DMatrix(data = as.matrix(data))


In [45]:
xgbregmin=predict(xgb_min_reg,alldataxgb)
xgbclsmin=predict(xgb_min_cl,alldataxgb)
lgbregmin=predict(lgb_min_reg,as.matrix(data))
lgbclsmin=predict(lgb_min_cl,as.matrix(data))

In [46]:
xgbregmin

[1]  3.999790e+00  4.000044e+00  2.461423e+00  1.000031e+00  1.517344e+00
  [6]  1.038698e+00  8.636927e-01  1.000020e+00  8.026880e-01  2.003150e+00
 [11]  2.412664e+00  8.743667e-01  3.998688e+00  4.001712e+00  3.191036e+00
 [16]  4.325136e+00  3.001709e+00  3.999686e+00  1.999669e+00  2.096856e+00
 [21]  1.360732e+00  1.452837e+00  2.229420e+00  9.932319e-01  9.046938e-01
 [26] -6.973743e-06  1.813601e+00  1.975116e+00  1.566831e+00  9.122193e-04
 [31]  1.096935e+00  9.855031e-01  1.000154e+00  3.150064e-01  3.844066e-01
 [36]  6.858784e-01  3.999741e+00  4.273013e+00  2.427876e+00  2.999032e+00
 [41]  3.000411e+00  3.032185e+00  3.030865e+00  2.495029e+00  2.377814e+00
 [46]  4.009754e+00  3.078099e+00  2.744858e+00  4.000101e+00  4.000101e+00
 [51]  4.000012e+00  3.999837e+00  2.999696e+00  1.000818e+00  3.335005e+00
 [56]  3.999849e+00  2.000757e+00  7.906258e-04  1.553507e-01  9.998122e-01
 [61]  2.756338e+00  2.333839e+00  2.013849e+00  3.996022e+00  3.938352e+00
 [66]  3.000802e+00  2.809384e+00  2.188851e+00  1.999489e+00  3.050325e-01
 [71]  2.440184e-02  2.939105e-04 -1.010931e-02  3.374770e+00  3.275003e+00
 [76]  2.405354e+00  1.478758e+00  2.028962e+00  1.426172e+00  2.999947e+00
 [81]  3.708298e+00  3.255411e+00  3.182873e+00  3.227401e+00  2.999059e+00
 [86]  3.857224e+00  1.599663e+00  4.000022e+00  1.000228e+00  2.699595e+00
 [91]  3.805874e+00  3.999697e+00  3.000498e+00  2.229929e+00  1.646646e+00
 [96]  3.998850e+00  1.007130e+00  1.999260e+00  3.999871e+00  2.825104e+00
[101] -1.087189e-03  4.000979e+00  3.000358e+00  5.956888e-04  3.696776e+00
[106]  3.041350e+00  3.111183e+00  3.747866e+00  3.999725e+00  3.963427e+00
[111]  1.599663e+00  3.775877e+00  3.000482e+00  2.000179e+00  3.755990e+00
[116]  3.999479e+00  3.604801e+00  2.748360e+00  1.874238e-03  1.011245e+00
[121]  2.000731e+00  7.493496e-04  1.052424e+00  4.243595e+00  2.999122e+00
[126]  1.379758e-03  3.668120e+00  3.945959e+00 -2.841055e-03  1.000531e+00
[131]  2.999820e+00  3.334704e+00  2.876829e+00  2.760663e+00  3.669433e+00
[136]  3.999785e+00  3.866132e+00  2.999290e+00  2.617820e+00  2.899951e+00
[141]  3.841743e+00  3.999851e+00  3.000275e+00  3.424150e+00  3.000455e+00
[146]  3.392110e+00  6.326160e-01  1.800358e-04  4.000108e+00  2.999924e+00
[151]  3.779698e+00  3.866991e+00  3.998500e+00  2.000254e+00  2.733846e+00
[156]  2.401229e+00  2.240493e+00  3.308062e+00  3.999508e+00  3.930691e+00
[161]  3.011366e+00  3.454726e+00  1.841725e+00  3.998498e+00 -3.064275e-04
[166]  2.291203e-04  9.295046e-04  3.892838e+00  3.667532e+00  3.538883e+00
[171]  3.552156e+00  2.198707e+00  6.222549e-01  3.413951e+00  2.901889e+00
[176]  4.001660e+00  2.245202e+00  3.389700e+00  2.911957e+00  3.486036e+00
[181]  2.001121e+00  1.817104e+00  2.871500e+00  3.691007e+00  2.999985e+00
[186]  3.998400e+00  2.000769e+00  2.000220e+00  3.999608e+00  2.039845e+00
[191]  1.886918e+00  2.467196e+00  3.000140e+00  1.737398e+00  2.714002e+00
[196]  3.998864e+00  2.000431e+00  3.858430e+00  2.999908e+00  2.282202e+00
[201]  3.000121e+00  2.965866e+00  3.084669e+00  2.601964e+00  3.999404e+00
[206]  3.018279e+00  2.970860e+00  3.296680e+00  4.000641e+00  3.369107e+00
[211]  3.999064e+00  2.940515e+00  1.999711e+00  2.999687e+00  3.999139e+00
[216]  3.920789e+00  1.825840e+00  1.966160e+00  1.636472e+00  4.000229e+00
[221]  3.362664e+00  3.291262e+00  3.394128e+00  1.766617e+00  6.790161e-04
[226]  2.062305e+00  9.993039e-01  1.653403e-03 -6.459951e-04  1.999389e+00
[231]  1.457929e+00  2.000788e+00  2.526332e+00  9.464124e-01  9.999598e-01
[236]  1.999829e+00  3.998960e+00  1.899347e+00  1.744686e+00  1.856166e+00
[241]  9.998187e-01  1.553252e+00  1.460041e+00  5.357876e-01  1.000640e+00
[246]  4.718533e-01  1.846090e+00  3.999677e+00  4.000442e+00  3.999295e+00
[251]  2.999949e+00  3.054992e+00  3.999641e+00  3.999992e+00  2.214751e+00
[256]  3.221290e+00  2.000184e+00  1.999730e+00  3.598310e+00  2.038478e-01
[261]  3.999747e+00  3.324853e+00  3.997460e+00  8

In [47]:
xgbclsmin

[1] 4 4 3 1 4 4 2 1 1 2 3 0 4 4 4 4 3 4 2 4 2 4 4 2 1 0 2 3 0 0 0 1 1 3 0 0 4
 [38] 4 3 3 3 3 3 3 2 4 3 2 4 4 4 4 3 1 3 4 2 0 0 1 4 2 4 4 4 3 4 2 2 2 0 0 0 4
 [75] 4 3 3 4 3 3 4 4 3 4 3 3 4 4 1 3 4 4 3 0 0 4 3 2 4 3 0 4 3 0 4 3 4 4 4 4 4
[112] 4 3 2 4 4 4 3 0 0 2 0 2 4 3 0 4 4 0 1 3 4 4 4 4 4 4 3 2 2 4 4 3 3 3 4 0 0
[149] 4 3 4 4 4 2 2 1 3 4 4 4 3 4 4 4 0 0 0 4 4 4 4 4 0 4 4 4 1 4 4 4 2 4 4 4 3
[186] 4 2 2 4 3 3 3 3 3 3 4 2 4 3 3 3 3 3 3 4 3 3 4 4 4 4 2 2 3 4 4 2 2 1 4 3 4
[223] 4 3 0 4 1 0 0 2 0 2 4 2 1 2 4 2 4 3 1 3 2 0 1 1 0 4 4 4 3 3 4 4 2 4 2 2 3
[260] 0 4 3 4 4 0 0 0 4 4 3 3 4 4 4 4 4 2 4 3 2 4 2 0 3 4 4 3 4 4 4 4 4 2 1 0 4
[297] 3 4 2 3 4 4 4 3 3 4 3 4 4 2 3 3 3 3 3 4 3 3 3 4 3 3 3 4 4 3 4 2 4 4 4 3 4
[334] 4 3 2 3 3 2 3 4 4 4 2 4 4 3 3 4 3 2 0 3 3 1 3 4 4 4

In [48]:
lgbregmin

[1]  4.08094042  3.98200270  3.17526746  0.94374156  1.31342575  2.42129688
  [7]  1.55393031  0.93843847  0.56528517  2.05334350  3.03517603  1.65287101
 [13]  3.97311659  4.07254470  2.17644718  4.65518960  3.11412940  3.96185422
 [19]  1.91161425  2.57916842  1.63618343  1.62582650  2.47829296  1.73251001
 [25]  1.17284525  0.09056126  0.79999392  1.47973572  1.62069237  0.16247369
 [31]  0.03216223 -0.16642604  0.99580685  1.33276918  1.07571072  1.01364820
 [37]  3.97426974  4.44652117  3.06012361  2.83471454  3.09390695  3.05277065
 [43]  3.07471368  2.93761912  1.76972211  4.11398541  2.17128176  2.24540342
 [49]  4.00358675  4.00057959  4.00092523  3.98232580  2.95144556  1.12823473
 [55]  2.57611958  3.91734628  2.05854902  0.07404203  1.75876389  0.98660346
 [61]  2.36973961  2.13499915  3.10397347  3.35781261  3.62934991  3.03458051
 [67]  2.71211234  2.61416244  1.90274862  1.09173687  0.23411965  0.03135112
 [73] -0.05854094  1.82065132  3.29720517  2.92093196  1.22293112  2.58696468
 [79]  1.49155504  2.92851243  3.57234821  3.20509948  3.11091470  3.54987760
 [85]  2.93074441  4.11831202  3.27690758  3.99467254  0.99961601  3.05441193
 [91]  3.21463865  4.06543968  3.04586435  1.17564057  2.28037276  3.91746548
 [97]  0.71416836  1.85481215  3.71892826  3.08331374  0.05646766  4.02444957
[103]  3.01829066  0.07695276  2.36523537  3.01060856  2.59939109  3.95111437
[109]  3.88744095  4.80754843  3.27690758  2.99747559  3.02919327  1.85580871
[115]  3.25666735  3.91092968  3.91755881  2.78237171  0.21950904  2.84967060
[121]  2.10412470  0.13186591  1.41953899  3.91021018  2.93759921 -0.02020734
[127]  3.23730335  4.30352135  0.16900602  1.26317907  2.96670109  2.97263048
[133]  2.38968465  2.37064814  4.43522582  3.97600188  2.96933055  3.02449552
[139]  3.13441922  1.50128569  3.30885400  4.02747701  3.01947347  3.90951769
[145]  3.02878254  3.06252208  0.98191648 -0.01303285  3.99459331  3.05360695
[151]  3.83062359  3.46787190  3.92391882  2.11430888  2.38307772  1.41024873
[157]  2.68532777  2.56381832  4.11865741  3.86166926  2.98477000  3.67345513
[163]  2.34204101  3.92516560  0.06899996  0.02243592  0.22047066  3.45851030
[169]  3.64422433  3.54784113  3.39397093  3.47596711  1.01493353  3.29852625
[175]  2.15509107  4.04674344  2.25484967  1.08230790  2.62470758  3.21367336
[181]  2.13551938  3.15913366  2.16081115  2.77161055  2.84427166  3.97873270
[187]  2.08548978  1.98874215  3.99418229  2.45547970  2.56886606  3.22111085
[193]  2.98778392  0.58296824  2.86869043  3.98901317  2.06364406  3.19102436
[199]  2.92425263  1.34658939  2.97388789  3.08794970  2.76259703  2.89783390
[205]  3.95153104  2.55531619  3.95454278  2.65068306  4.04426660  2.35492990
[211]  3.97645408  2.58491426  2.06198880  3.00689482  3.89349868  3.77680314
[217]  1.48481384  1.87275177  3.00375490  4.04414324  3.75628521  2.72643529
[223]  3.89115315  1.57853641  0.12384252  2.27823350  1.00021656  0.14811583
[229] -0.04137028  1.98294609  1.09518691  2.10302478  2.75415000  0.89639610
[235]  0.95334618  1.94506025  3.94152971  2.82068462  1.81541368  2.42330089
[241]  0.86341815  2.40843462  1.34786914  0.20419780  1.04530579  0.02144835
[247]  0.37720299  4.03334426  3.95512443  3.96321929  2.94983007  2.68436567
[253]  3.96322447  3.98163968  2.63699963  3.90374485  2.02604447  1.99539545
[259]  2.57725095  0.82395237  3.93205421  2.32303916  3.70957348  1.02009117
[265]  1.49916295  1.51448237  0.10375053  3.79082812  1.76942732  2.80972987
[271]  2.91315716  3.38099570  3.69945221  3.89924648  3.30577186  2.33422941
[277]  1.38628197  4.03217602  2.86862270  2.02691212  3.89157037  1.88108978
[283]  0.10586786  2.99836838  4.05320657  3.07022398  3.44608341  3.71159750
[289]  3.95350109  4.00608000  3.99851552  3.90901208  1.95159454  1.48839077
[295]  0.28950894  3.96610401  3.46594086  3.58513200  2.30623862  2.56266117
[301]  2.41612565  3.95669024  3.04387676  2.80975150  2.95181256  2.42964984
[307]  3.03044746  2.686

In [49]:
lgbclsmin

2.451516e-07,1.122478e-08,7.671836e-07,1.130175e-06,9.999978e-01
1.976005e-07,4.757265e-08,1.079353e-07,2.640691e-05,9.999732e-01
9.573086e-04,1.964570e-05,1.971077e-05,9.813689e-01,1.763444e-02
1.981848e-05,9.999791e-01,1.064182e-06,2.597846e-08,3.196668e-08
3.384949e-01,4.624901e-01,4.548924e-04,2.242255e-09,1.985601e-01
2.469650e-02,4.074564e-06,3.157445e-04,2.156642e-10,9.749837e-01
1.487031e-03,5.339253e-02,9.340553e-01,3.070061e-03,7.995083e-03
8.971598e-06,9.999734e-01,1.742311e-05,2.337754e-07,5.226801e-11
2.084093e-03,9.978432e-01,9.099267e-09,7.274493e-05,6.147425e-10
6.337980e-11,1.327372e-10,9.999593e-01,2.140961e-08,4.071347e-05
5.789156e-04,8.875320e-02,5.449030e-07,9.102283e-01,4.390041e-04


In [50]:
datalength=dim(data)[1]
lgbclsm=matrix(lgbclsmin,m,datalength)
lgbclsminr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsminr[i,]=which.max(lgbclsm[,i])
}
lgbclsminr=lgbclsminr-1

In [51]:
lgbclsminr

4
0
1
3
4
0
0
0
1
1
4


In [52]:
xgbregmean=predict(xgb_mean_reg,alldataxgb)
xgbclsmean=predict(xgb_mean_cl,alldataxgb)
lgbregmean=predict(lgb_mean_reg,as.matrix(data))
lgbclsmean=predict(lgb_mean_cl,as.matrix(data))

In [53]:
lgbclsm=matrix(lgbclsmean,m,datalength)
lgbclsmeanr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsmeanr[i,]=which.max(lgbclsm[,i])
}
lgbclsmeanr=lgbclsmeanr-1

In [54]:
preallmin=cbind(xgbclsmin,xgbregmin)
preallmin=cbind(preallmin,lgbclsminr)
preallmin=cbind(preallmin,lgbregmin)

In [55]:
colnames(preallmin)=c('xgbclsmin','xgbregmin','lgbclsmin','lgbregmin')
head(preallmin)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
4,3.999790,4,4.0809404
4,4.000044,0,3.9820027
3,2.461423,1,3.1752675
1,1.000031,3,0.9437416
4,1.517344,4,1.3134258
4,1.038698,0,2.4212969


In [56]:
preallmean=cbind(xgbclsmean,xgbregmean)
preallmean=cbind(preallmean,lgbclsmeanr)
preallmean=cbind(preallmean,lgbregmean)

In [57]:
head(preallmean)

xgbclsmean,xgbregmean,,lgbregmean
4,3.9992321,4,4.0544632
4,3.9998746,0,4.0330911
4,1.4142292,1,3.3061475
1,0.9998934,3,1.0050613
0,0.3454669,0,-0.1948324
0,1.4086787,4,1.5244908


In [58]:
colnames(preallmean)=c('xgbclsmean','xgbregmean','lgbclsmean','lgbregmean')
head(preallmean)

xgbclsmean,xgbregmean,lgbclsmean,lgbregmean
4,3.9992321,4,4.0544632
4,3.9998746,0,4.0330911
4,1.4142292,1,3.3061475
1,0.9998934,3,1.0050613
0,0.3454669,0,-0.1948324
0,1.4086787,4,1.5244908


In [59]:
end_time = Sys.time()
time_matrix[4,]=end_time-start_time

In [60]:
result_list=list(preallmin,preallmean,time_matrix)

In [66]:
save(result_list, file = "Weekly_FForma_opt_pre_result.RData")

In [58]:
time_matrix

user_time,system_time,elapsed_time
4.6199644,4.6199644,4.6199644
3.5019691,3.5019691,3.5019691
2.8674052,2.8674052,2.8674052
0.8890216,0.8890216,0.8890216


In [61]:
nnetarl=load('Weekly_FForma_datatestlist.RData')
nnetar_datalist <- eval(parse(text = nnetarl))

In [62]:
nnetar_predh=nnetar_datalist[[2]]
nnetar_pred_res=nnetar_datalist[[1]]

In [63]:
MASE=nnetar_pred_res[,,,6]
m=5

In [64]:
realbestmin=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    min_value=apply(count,1,min)
    if (max(min_value,na.rm = TRUE)==0){
        realbestmin[i,]=0
        }
    else
        {
        min_value[is.na(min_value)]=100 
        realbestmin[i,]= whichmin(min_value)
    }
    }

In [65]:
realbestmean=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    mean_value=apply(count,1,meanunique)
    if (max(mean_value,na.rm = TRUE)==0){
        realbestmean[i,]=0
        }
    else
        {
        mean_value[is.na(mean_value)]=100 
        realbestmean[i,]= whichmin(mean_value)
    }
    }

In [66]:
sum(lgbclsmeanr[index==3]==realbestmean)/length(realbestmean)

[1] 0.2280702

In [67]:
mean(abs(lgbclsmeanr[index==3] -realbestmean))

[1] 1.508772